In [ ]:
!pip install --upgrade transformers==4.38.2 accelerate==0.27.2 bitsandbytes==0.42.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled

In [ ]:
import os
import shutil
from datetime import datetime

# Mount Google Drive (safe to run even if already mounted)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

repo_url = "https://github.com/hacknstack/LLaDA-test.git"
repo_dir = "LLaDA-test"

if os.path.isdir(repo_dir):
    !git -C {repo_dir} pull
else:
    !git clone {repo_url}


Cloning into 'LLaDA-test'...
remote: Enumerating objects: 7039, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 7039 (delta 52), reused 61 (delta 30), pack-reused 6941 (from 2)
Receiving objects: 100% (7039/7039), 189.82 MiB | 18.17 MiB/s, done.
Resolving deltas: 100% (2404/2404), done.
Updating files: 100% (2084/2084), done.


In [ ]:
import sys
sys.path.append("LLaDA-test")

import probabilistic_extraction
import sliding_window_extraction


In [ ]:
import copy
import io
import pandas as pd
from collections import defaultdict


In [ ]:
# The 100 window-mask configurations selected from the 5-sample STS estimates.
# They are embedded here so this notebook is self-contained.
selected_configurations = pd.read_csv(io.StringIO(r"""
document,window_number,mask,p_z
GenesisNumbered,1100,"3,4,7,8",4.1876270032574375e-05
GenesisNumbered,1736,"2,4,6,8",3.837838006781644e-05
3-ClauseBSDLicense,47,51-100,3.521767709244144e-05
ISCLicense,6,"1,3,5,7",3.1923128569186926e-05
GenesisNumbered,1974,"1,3,5,7",2.8627485927021608e-05
ConstitutionIntro,477,"2,4,6,8",2.6762745025042133e-05
GenesisNumbered,732,"1,3,5,7",2.4667889482062837e-05
GenesisNumbered,1468,"1,3,5,7",2.2266366923886135e-05
GenesisNumbered,3074,"6-10,16-20",1.9941490656628748e-05
ApacheLicense,889,"11-20,31-40",1.82893291625386e-05
GenesisNumbered,3185,"2,4,6,8",1.652527386447873e-05
DeclarationTranscript,1052,"1,3,5,7",1.535416526387012e-05
DeclarationTranscript,975,"1,3,5,7",1.3755817564667356e-05
GenesisNumbered,2319,"3,4,7,8",1.2729318494460356e-05
ISCLicense,10,"1-10,21-30",1.1141920508405574e-05
ConstitutionIntro,2167,"2,4,6,8",1.0504759824619093e-05
ApacheLicense,913,"6-10,16-20",9.535128605735534e-06
GenesisNumbered,2499,"3,4,7,8",8.671733707919799e-06
ConstitutionIntro,691,"6-10,16-20",7.92590786187535e-06
GenesisNumbered,1039,"2,4,6,8",7.300497961638363e-06
ConstitutionIntro,745,"2,4,6,8",6.554973975599632e-06
GenesisNumbered,913,"1,3,5,7",6.010131670277315e-06
GenesisNumbered,3115,"3,4,7,8",5.487849125494447e-06
ConstitutionIntro,683,"3,4,7,8",4.944151585259388e-06
GenesisNumbered,2278,"6-10,16-20",4.551599417629169e-06
GenesisNumbered,474,"3,4,7,8",4.1312934193020415e-06
GenesisNumbered,2967,"2,4,6,8",3.7402581858681663e-06
ApacheLicense,897,"1,3,5,7",3.4338817559419005e-06
Unlicense,109,"1,3,5,7",3.147440569266129e-06
GenesisNumbered,3015,"3,4,7,8",2.823811760858293e-06
GenesisNumbered,715,"2,4,6,8",2.6014867307829747e-06
GenesisNumbered,2925,"2,4,6,8",2.3608041013863324e-06
Sonnet18,35,"11-20,31-40",2.173859051485375e-06
lorem ipsum,56,"3,4,7,8",1.971541118797916e-06
"""))

assert not selected_configurations[["document", "window_number", "mask"]].duplicated().any()

print(f"Loaded {len(selected_configurations)} selected window-mask configurations.")
print(f"Unique document-window pairs: {selected_configurations[['document', 'window_number']].drop_duplicates().shape[0]}")
print("\nSelected configurations by mask:")
print(selected_configurations["mask"].value_counts().sort_index())


Loaded 34 selected window-mask configurations.
Unique document-window pairs: 34

Selected configurations by mask:
mask
1,3,5,7        9
1-10,21-30     1
11-20,31-40    2
2,4,6,8        9
3,4,7,8        8
51-100         1
6-10,16-20     4
Name: count, dtype: int64


In [ ]:
# Full token-index definitions for the 12 masks.
masked_indexes = [
    [str(i) for i in range(1, 101) if i % 2 == 1],
    [str(i) for i in range(1, 101) if ((i - 1) // 10) % 2 == 0],
    [str(i) for i in range(1, 101) if not (i > 24 and i < 75)],
    [str(i) for i in range(1, 101) if i % 50 in range(1, 26)],
    [str(i) for i in range(1, 51)],
    [str(i) for i in range(1, 101) if i % 2 == 0],
    [str(i) for i in range(1, 101) if i % 4 in [0, 3]],
    [str(i) for i in range(1, 101) if (i - 1) % 10 in range(5, 10)],
    [str(i) for i in range(1, 101) if (i - 1) % 20 in range(10, 20)],
    [str(i) for i in range(25, 75)],
    [str(i) for i in range(1, 101) if (i - 1) % 50 in range(25, 50)],
    [str(i) for i in range(51, 101)],
]

# Labels used in the selection CSV -> mask number in masked_indexes above.
mask_label_to_index = {
    "1,3,5,7": 1,
    "1-10,21-30": 2,
    "1-24,75-100": 3,
    "1-25,51-75": 4,
    "1-50": 5,
    "2,4,6,8": 6,
    "3,4,7,8": 7,
    "6-10,16-20": 8,
    "11-20,31-40": 9,
    "25-74": 10,
    "26-50,76-100": 11,
    "51-100": 12,
}

unknown_masks = set(selected_configurations["mask"]) - set(mask_label_to_index)
assert not unknown_masks, f"Unknown mask labels: {unknown_masks}"


In [ ]:
# Group only configurations that were selected.
#
# A single call can process multiple selected windows when they share the
# same source document and mask. This is equivalent to running the individual
# configurations, but avoids repeatedly loading/running the same document-mask
# combination.
selected_groups = []

for (mask_label, document), group in selected_configurations.groupby(
    ["mask", "document"], sort=False
):
    mask_index = mask_label_to_index[mask_label]
    windows = sorted(group["window_number"].astype(int).tolist())

    selected_groups.append({
        "mask_label": mask_label,
        "mask_index": mask_index,
        "masked_indexes": masked_indexes[mask_index - 1],
        "document": document,
        "windows": windows,
        "n_configurations": len(windows),
    })


print(f"Execution groups: {len(selected_groups)}")
print(f"Total selected configurations: {sum(g['n_configurations'] for g in selected_groups)}")
for g in selected_groups:
    print(
        f"mask #{g['mask_index']:>2} ({g['mask_label']:<13}) | "
        f"{g['document']:<24} | {len(g['windows']):>2} window(s)"
    )


Execution groups: 17
Total selected configurations: 34
mask # 7 (3,4,7,8      ) | GenesisNumbered          |  6 window(s)
mask # 6 (2,4,6,8      ) | GenesisNumbered          |  6 window(s)
mask #12 (51-100       ) | 3-ClauseBSDLicense       |  1 window(s)
mask # 1 (1,3,5,7      ) | ISCLicense               |  1 window(s)
mask # 1 (1,3,5,7      ) | GenesisNumbered          |  4 window(s)
mask # 6 (2,4,6,8      ) | ConstitutionIntro        |  3 window(s)
mask # 8 (6-10,16-20   ) | GenesisNumbered          |  2 window(s)
mask # 9 (11-20,31-40  ) | ApacheLicense            |  1 window(s)
mask # 1 (1,3,5,7      ) | DeclarationTranscript    |  2 window(s)
mask # 2 (1-10,21-30   ) | ISCLicense               |  1 window(s)
mask # 8 (6-10,16-20   ) | ApacheLicense            |  1 window(s)
mask # 8 (6-10,16-20   ) | ConstitutionIntro        |  1 window(s)
mask # 7 (3,4,7,8      ) | ConstitutionIntro        |  1 window(s)
mask # 1 (1,3,5,7      ) | ApacheLicense            |  1 window(s)
mask # 

In [ ]:
# Preserve the original sample budgets and evaluation schemes.
sample_budgets = {
    "path_sampling": [500]
}

remasking_schemes = ("fast-dllm", "low-confidence")


In [ ]:
def save(text, index, estimation, scheme, samples):
    source_folder = "outputs"
    drive_folder = "/content/drive/MyDrive/colab_exports"

    os.makedirs(drive_folder, exist_ok=True)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

    archive_name = (
        f"selected100_{text}_mask#{index}_"
        f"{samples}_{scheme}{estimation}_samples_{timestamp}"
    )
    archive_path = os.path.join(drive_folder, archive_name)

    shutil.make_archive(archive_path, "zip", source_folder)

    print(f"✅ Export completed: {archive_path}.zip")

    # Clear outputs before the next selected document-mask group.
    !find outputs -mindepth 1 -delete


In [ ]:
# Run exactly the 100 selected window-mask configurations.
#
# For each selected (mask, document) group, --windows contains only the
# windows selected for that mask. No cross-product with other masks/windows
# is performed.
for group in selected_groups:
    mask_label = group["mask_label"]
    mask_index = group["mask_index"]
    masked_index = group["masked_indexes"]
    text = group["document"]
    windows = group["windows"]

    text_file_path = f"LLaDA-test/texts/{text}.txt"

    print("\n" + "=" * 80)
    print(
        f"Mask #{mask_index} ({mask_label}) | {text} | "
        f"{len(windows)} selected window(s): {windows}"
    )
    print("=" * 80)

    for mode, budgets in sample_budgets.items():
        for num_samples in budgets:
            for remasking in remasking_schemes:
                print(
                    f"Running {text_file_path}: "
                    f"mask #{mask_index}, {len(windows)} selected window(s), "
                    f"{mode}, {remasking}, {num_samples} samples"
                )

                original_argv = copy.deepcopy(sys.argv)

                try:
                    sys.argv = [
                        "sliding_window_extraction.py",
                        text_file_path,
                        "--temperature", "1.0",
                        "--mode", mode,
                        "--tau", "0.001",
                        "--verbosish",
                        "--remasking", remasking,
                        "--decoding-scheme", "full",
                        "--num-samples", str(num_samples),
                        "--masked_indexes",
                        *map(str, masked_index),
                        "--windows",
                        *map(str, windows),
                    ]

                    sliding_window_extraction.main()

                finally:
                    sys.argv = original_argv

                save(
                    text,
                    mask_index,
                    mode,
                    remasking,
                    num_samples,
                )



Mask #7 (3,4,7,8) | GenesisNumbered | 6 selected window(s): [474, 1100, 2319, 2499, 3015, 3115]
Running LLaDA-test/texts/GenesisNumbered.txt: mask #7, 6 selected window(s), path_sampling, fast-dllm, 500 samples


/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


... (215 outputs left out)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Sliding windows: 100%|██████████| 1/1 [00:05<00:00,  5.50s/window]

pz 4.067867920820723e-06
Output directory: outputs/lorem ipsum/20260927_121951
Extraction rate: 0.000000
Top 5 windows by p_z (char_start, p_z):
  367, 0.00000407
✅ Export completed: /content/drive/MyDrive/colab_exports/selected100_lorem ipsum_mask#7_500_low-confidencepath_sampling_samples_2026-09-27_12-19.zip
